In [6]:
!pip install pydub mutagen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 1.5 MB/s eta 0:00:00


In [7]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [9]:
import os
import shutil
import zipfile
from pathlib import Path
from pydub import AudioSegment
from mutagen.mp3 import MP3
from google.colab import files

def convert_and_analyze_audio(file_list=None):
    """
    在 Google Colab 中將 M4A 檔案轉為 MP3（保持相同檔名）。
    1. 若清單為空，自動彈出網頁版圖形選單 (GUI)。
    2. 每轉換好一個 MP3，即時偵測並顯示其音質數據。
    3. 全部轉換完成後，自動打包所有 MP3 並觸發瀏覽器下載。
    """
    # 確保 Colab 環境中有系統轉碼核心 FFmpeg
    if shutil.which("ffmpeg") is None:
        print("🔧 偵測到環境缺少 FFmpeg，正在為您自動安裝...")
        !apt-get update -qq && apt-get install -y ffmpeg > /dev/null
        print("✅ FFmpeg 安裝完成！\n")

    # 1. 檢查清單是否為空：若為空，啟動 Colab 網頁圖形化檔案選取Popup
    if not file_list:
        print("💡 傳入清單為空。正在拉起圖形化檔案選擇選單...")
        print("👉 請點擊下方按鈕，選取本機電腦中的 .m4a 檔案進行上傳（可按住 Ctrl/Shift 多選）：\n")

        # 彈出網頁版檔案選取介面
        uploaded = files.upload()

        # 篩選出使用者剛才成功上傳的所有 .m4a 檔案
        file_list = [filename for filename in uploaded.keys() if filename.lower().endswith('.m4a')]

        if not file_list:
            print("❌ 您關閉了視窗或未選擇任何有效的 .m4a 檔案，轉換終止。")
            return []

    # 2. 開始批次轉換與音質分析
    successful_conversions = []
    print(f"\n🚀 正在開始處理 {len(file_list)} 個音訊檔案...")
    print("-" * 60)

    for file_path in file_list:
        path = Path(file_path)

        if not path.is_file():
            print(f"⚠️ 略過：找不到檔案 '{path}'")
            continue

        try:
            print(f"🎵 正在轉碼中: {path.name}")

            # 載入原始 M4A 音訊
            audio = AudioSegment.from_file(path, format="m4a")

            # 建立同路徑、同檔名、僅改副檔名為 .mp3 的輸出路徑
            output_path = path.with_suffix('.mp3')

            # 匯出為 MP3（此處可加上 bitrate="320k" 手動調高目標音質）
            audio.export(output_path, format="mp3")
            print(f"   💾 成功儲存: {output_path.name}")

            # 【核心功能】即時偵測剛轉好的 MP3 音質
            audio_info = MP3(output_path).info
            bitrate_kbps = int(audio_info.bitrate / 1000)
            sample_rate_hz = audio_info.sample_rate
            channels = "立體聲 (Stereo)" if audio_info.channels == 2 else "單聲道 (Mono)"

            print(f"   📊 偵測音質 -> 碼率: {bitrate_kbps} kbps | 取樣率: {sample_rate_hz} Hz | 聲道: {channels}")
            print("-" * 60)

            successful_conversions.append(output_path)

        except Exception as e:
            print(f"❌ 檔案 {path.name} 處理時發生錯誤: {e}")
            print("-" * 60)

    # 3. 全部轉換完成後，自動打包所有 MP3 並下載
    if successful_conversions:
        zip_filename = "converted_mp3_files.zip"
        print(f"\n📦 正在將 {len(successful_conversions)} 個 MP3 檔案打包壓縮中...")

        # 建立壓縮檔
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file in successful_conversions:
                zipf.write(file, arcname=file.name)

        print(f"⬇️ 壓縮完成！正在透過瀏覽器將 '{zip_filename}' 下載至您的電腦...")
        files.download(zip_filename)
        print("🎉 任務全數執行完畢！")
    else:
        print("\n❌ 沒有成功轉換任何檔案，取消打包下載流程。")

    return [str(p) for p in successful_conversions]


# --- 執行範例 ---
if __name__ == "__main__":

    # 傳入空清單，觸發全自動的「網頁 GUI 彈窗上傳 -> 轉檔 -> 音質解析 -> 打包下載」一條龍服務
    empty_list = []
    convert_and_analyze_audio(empty_list)


💡 傳入清單為空。正在拉起圖形化檔案選擇選單...
👉 請點擊下方按鈕，選取本機電腦中的 .m4a 檔案進行上傳（可按住 Ctrl/Shift 多選）：



Saving 01-NV GPU Evolution History-The 1000x leap from Fermi to Blackwell.m4a to 01-NV GPU Evolution History-The 1000x leap from Fermi to Blackwell.m4a
Saving 02.1-CUDA Intro, Shared Mem-The Brutal Logistics of GPU Memory.m4a to 02.1-CUDA Intro, Shared Mem-The Brutal Logistics of GPU Memory.m4a
Saving 02.2-CUDA Opt-Hiding Memory Latency in CUDA Kernels.m4a to 02.2-CUDA Opt-Hiding Memory Latency in CUDA Kernels.m4a
Saving 02.3-CUDA Atomic, Concurrency-Choreographing Parallel GPU Data Flow.m4a to 02.3-CUDA Atomic, Concurrency-Choreographing Parallel GPU Data Flow.m4a
Saving 02.4-CUDA Mem Mgmt-Shared versus Unified CUDA Memory Architectures.m4a to 02.4-CUDA Mem Mgmt-Shared versus Unified CUDA Memory Architectures.m4a
Saving 02.4-CUDA 共享記憶體與 Bank 衝突.m4a to 02.4-CUDA 共享記憶體與 Bank 衝突.m4a
Saving 02.5-CUDA Perf Tuning-Fixing NVIDIA GPU Performance Bottlenecks.m4a to 02.5-CUDA Perf Tuning-Fixing NVIDIA GPU Performance Bottlenecks.m4a
Saving 02.5-Perf Tuning-破解 GPU 運算停滯的剖析實務.m4a to 02.5-Perf Tuni

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 任務全數執行完畢！
